# SWAN-SF Partition 1 — Combined Dataset Construction

This notebook converts the raw Partition 1 MVTS instance files (one file per active-region observation window, split across `FL/` and `NF/` folders) into a single combined dataset ready for sequential modeling:

- **`X`** — a 3D NumPy tensor of shape **`(N, 60, F)`**, where `N` is the number of instances in Partition 1, `60` is the fixed 12-hour / 12-minute-cadence window length, and `F` is the per-timestep feature depth (reported at runtime — see the note below).
- **Side-car arrays** — 1D arrays, one entry per instance, carrying the FL/NF label, flare class, HARP number, source filename, and two data-quality flags, kept *parallel* to `X` rather than baked into it (per the "side-car tracking" decision).

It implements the cleaning policy established during EDA (`swan-sf_missing_data_quality_findings.md`):

| Issue | Handling |
|---|---|
| 8 `_LABEL` columns | Dropped — structurally sparse JSON annotations, not usable features |
| Eclipse blackout (`QUALITY` NaN / `SPEI` False) | Linear interpolation, **per file**, `limit_direction='both'` |
| Untrusted-but-present rows (`IS_TMFI` False) | Kept as-is; `IS_TMFI` retained as a feature, not imputed |
| Degraded `XRQUALITY` | Flagged at the **instance level** (not dropped) for label-trust filtering later |

**A note on feature count:** dropping just the 8 `_LABEL` columns leaves ~47 raw numeric columns (55 total − 8). This notebook also adds one engineered column, `was_interpolated`, which the findings doc suggested so models can discount eclipse-filled rows — so the final tensor will likely be `(N, 60, 48)`. The exact `F` is detected dynamically from the data rather than hardcoded (see Section 5), and is printed once the pipeline runs. Drop the last channel before modeling if you want to match the original `(N, 60, 47)` shape exactly.

**Before running:** check the paths in Section 2 match your Drive layout.

## 1. Imports

In [ ]:
import os
import re
import json

import numpy as np
import pandas as pd

try:
    from tqdm.notebook import tqdm
except ImportError:
    from tqdm import tqdm

pd.set_option('display.max_columns', 100)

## 2. Configuration

Edit these if your Drive layout differs:

- `drive_data_dir` — where the downloaded `partition1_instances.tar.gz` archive lives
- `local_extract_dir` — fast local Colab SSD path used for extraction
- `output_dir` — where the combined dataset gets written back to Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Where the partition archive currently lives
drive_data_dir = '/content/drive/MyDrive/solar_flare_forecasting/Data'

# True high-speed local Colab SSD path
local_extract_dir = '/content/solar_flare_data'

# Where the final combined dataset gets written back to Drive
output_dir = '/content/drive/MyDrive/solar_flare_forecasting/processed'

archive_name = 'partition1_instances.tar.gz'
archive_url = ('https://dataverse.harvard.edu/api/access/datafile/:persistentId'
               '?persistentId=doi:10.7910/DVN/EBCFKM/BMXYCB')

os.makedirs(local_extract_dir, exist_ok=True)
os.makedirs(output_dir, exist_ok=True)

## 3. Extract Partition 1 onto the Local SSD

In [ ]:
archive_path = os.path.join(drive_data_dir, archive_name)

if os.path.exists(archive_path):
    print(f"Unpacking {archive_name} into {local_extract_dir} ...")
    exit_code = os.system(f"tar -xzf '{archive_path}' -C '{local_extract_dir}'")
    if exit_code == 0:
        print("Extraction complete.")
    else:
        print(f"tar exited with code {exit_code} — check the archive path/permissions.")
else:
    print(f"Could not find archive at {archive_path}.")
    print("If you haven't downloaded it yet, fetch it from:")
    print(archive_url)

## 4. Discover All FL / NF Instance Files

We search recursively for folders literally named `FL` and `NF` rather than assuming a fixed nesting depth, since the exact folder structure after extraction can vary. Every file inside those folders is treated as one instance.

In [ ]:
def find_label_dirs(root, folder_name):
    return [d for d, dirs, files in os.walk(root) if os.path.basename(d) == folder_name]

fl_dirs = find_label_dirs(local_extract_dir, 'FL')
nf_dirs = find_label_dirs(local_extract_dir, 'NF')

print(f"FL directories found: {fl_dirs}")
print(f"NF directories found: {nf_dirs}")

file_list = []  # list of (filepath, 'FL'/'NF')
for d in fl_dirs:
    for fname in sorted(os.listdir(d)):
        fpath = os.path.join(d, fname)
        if os.path.isfile(fpath):
            file_list.append((fpath, 'FL'))
for d in nf_dirs:
    for fname in sorted(os.listdir(d)):
        fpath = os.path.join(d, fname)
        if os.path.isfile(fpath):
            file_list.append((fpath, 'NF'))

print(f"Total instance files found: {len(file_list)}")
print(f"  FL: {sum(1 for _, l in file_list if l == 'FL')}")
print(f"  NF: {sum(1 for _, l in file_list if l == 'NF')}")

## 5. Cleaning & Feature-Extraction Policy

A few deliberate choices, carried over from the EDA findings:

- **`LABEL_COLS`** — the 8 `_LABEL` columns, dropped outright.
- **`NO_INTERPOLATE`** — `XR_MAX`, `XRQUALITY`, `IS_TMFI`, `QUALITY`. These are independently-sourced GOES/quality fields, not smooth magnetic-field quantities, so they're excluded from the eclipse-blackout interpolation. `QUALITY` is itself the NaN flag during eclipse, so it gets an explicit sentinel (`-1`) instead of an interpolated bitmask — interpolating a bitmask wouldn't be physically meaningful.
- **Column classification is dynamic, not hardcoded.** `classify_columns()` splits each file's columns into numeric (tensor-eligible) vs. metadata (e.g. timestamp strings) by attempting numeric conversion, since exact column names/casing can shift slightly between dataset releases. `coerce_known_booleans()` runs first so that `IS_TMFI` / `SPEI`-style True/False columns are reliably picked up as numeric (0/1) regardless of how pandas inferred their dtype on read.
- The **first successfully-processed file** sets the canonical column order; every subsequent file is checked against it so the tensor's feature axis stays consistent across all ~73,000+ instances.

In [ ]:
LABEL_COLS = [
    'BFLARE_LABEL', 'CFLARE_LABEL', 'MFLARE_LABEL', 'XFLARE_LABEL',
    'BFLARE_LABEL_LOC', 'CFLARE_LABEL_LOC', 'MFLARE_LABEL_LOC', 'XFLARE_LABEL_LOC',
]

# Independently-sourced GOES/quality fields — excluded from eclipse-blackout
# interpolation (QUALITY is itself the NaN flag during eclipse, handled via
# a sentinel below instead).
NO_INTERPOLATE = ['XR_MAX', 'XR_QUAL', 'IS_TMFI', 'QUALITY']

EXPECTED_ROWS = 60

# QUALITY is a bitmask; NaN has no meaningful bitmask value, so eclipse-blackout
# rows get an explicit sentinel rather than a linearly-interpolated bitmask.
QUALITY_SENTINEL_FOR_ECLIPSE = -1

TRUE_TOKENS = {True, 'True', 'TRUE', 'true', 1, '1'}
FALSE_TOKENS = {False, 'False', 'FALSE', 'false', 0, '0'}


def coerce_known_booleans(df):
    """Force True/False-style columns to 0/1 ints before classification,
    so IS_TMFI / SPEI are reliably treated as numeric features regardless
    of how pandas inferred their dtype on read."""
    for col in df.columns:
        vals = set(df[col].dropna().unique().tolist())
        if vals and vals.issubset(TRUE_TOKENS | FALSE_TOKENS):
            df[col] = df[col].apply(
                lambda v: 1 if v in TRUE_TOKENS else (0 if v in FALSE_TOKENS else np.nan)
            )
    return df


def classify_columns(df):
    """Split columns into numeric (tensor-eligible) vs. metadata (e.g. a
    timestamp string column) without hardcoding exact column names."""
    numeric_cols, metadata_cols = [], []
    for col in df.columns:
        try:
            pd.to_numeric(df[col])
            numeric_cols.append(col)
        except (ValueError, TypeError):
            metadata_cols.append(col)
    return numeric_cols, metadata_cols


def parse_filename(filepath, fl_nf_label):
    """Decode '[FluxClass]@[GOESID]_..._ar[HARPNUM]_...' per the confirmed
    filename schema."""
    fname = os.path.basename(filepath)
    flux_match = re.match(r'^([A-Za-z0-9.]+)@', fname)
    flare_class = flux_match.group(1) if flux_match else fl_nf_label
    harp_match = re.search(r'ar(\d+)', fname)
    harpnum = int(harp_match.group(1)) if harp_match else -1
    return flare_class, harpnum

## 6. Per-File Processing Function

In [ ]:
# Locks in the canonical numeric-column order from the first file processed;
# every later file is checked against it.
column_reference = {'numeric_cols': None}


def process_file(filepath, fl_nf_label):
    df = pd.read_csv(filepath, sep='\t')
    df = df.drop(columns=[c for c in LABEL_COLS if c in df.columns])
    df = coerce_known_booleans(df)

    if len(df) != EXPECTED_ROWS:
        return None, f"unexpected row count: {len(df)} (expected {EXPECTED_ROWS})"

    numeric_cols, metadata_cols = classify_columns(df)

    if column_reference['numeric_cols'] is None:
        column_reference['numeric_cols'] = numeric_cols
    else:
        missing = [c for c in column_reference['numeric_cols'] if c not in numeric_cols]
        if missing:
            return None, f"missing expected columns: {missing}"
        numeric_cols = column_reference['numeric_cols']  # enforce canonical order

    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col])

    # --- Eclipse-blackout interpolation (Section 4 of the findings doc) ---
    interp_cols = [c for c in numeric_cols if c not in NO_INTERPOLATE]
    # Flag rows that actually need filling -- based on real nullity in the
    # interpolated feature columns themselves, NOT on the QUALITY flag.
    # (Full-partition validation showed QUALITY==NaN and the rows that truly
    # have missing magnetic-field values don't align 1:1, so gating on QUALITY
    # alone mislabels some genuinely-interpolated rows.)
    was_eclipse = df[interp_cols].isna().any(axis=1)
    n_interpolated = int(was_eclipse.sum())

    df[interp_cols] = df[interp_cols].interpolate(method='linear', limit_direction='both')
    if 'QUALITY' in df.columns:
        df['QUALITY'] = df['QUALITY'].fillna(QUALITY_SENTINEL_FOR_ECLIPSE)

    # --- Instance-level degraded-XRQUALITY flag (Section 4, third policy row) ---
    xrquality_degraded = bool((df['XR_QUAL'] < 12).any()) if 'XR_QUAL' in df.columns else False

    # --- Engineered flag so models can discount interpolated rows ---
    df['was_interpolated'] = was_eclipse.astype(int)
    final_cols = numeric_cols + ['was_interpolated']

    feature_matrix = df[final_cols].to_numpy(dtype=float)

    flare_class, harpnum = parse_filename(filepath, fl_nf_label)
    record = {
        'source_file': os.path.basename(filepath),
        'fl_nf_label': fl_nf_label,
        'flare_class': flare_class,
        'harpnum': harpnum,
        'n_interpolated_rows': n_interpolated,
        'xrquality_degraded': xrquality_degraded,
    }
    return feature_matrix, record

## 7. Run the Pipeline Over All of Partition 1

In [ ]:
all_features = []
records = []
skipped = []

for filepath, fl_nf_label in tqdm(file_list, desc='Processing Partition 1'):
    try:
        feature_matrix, result = process_file(filepath, fl_nf_label)
    except Exception as e:
        skipped.append({'source_file': filepath, 'reason': str(e)})
        continue
    if feature_matrix is None:
        skipped.append({'source_file': filepath, 'reason': result})
        continue
    all_features.append(feature_matrix)
    records.append(result)

print(f"Processed: {len(all_features)} instances")
print(f"Skipped:   {len(skipped)} files")
if skipped:
    print("First few skip reasons:")
    for s in skipped[:5]:
        print(' -', s)

## 8. Stack Into the Final Tensor + Side-Car Arrays

In [ ]:
X = np.stack(all_features)
meta_df = pd.DataFrame(records)
feature_cols = column_reference['numeric_cols'] + ['was_interpolated']

print("Tensor shape (N, 60, F):", X.shape)
print(f"Feature columns ({len(feature_cols)}):")
print(feature_cols)

## 9. Sanity Checks

In [ ]:
assert X.shape[1] == EXPECTED_ROWS, "Window length drifted from 60 — investigate before continuing."
assert X.shape[0] == len(meta_df), "Tensor / metadata row-count mismatch."

n_nan = int(np.isnan(X).sum())
print(f"Remaining NaNs in tensor: {n_nan}")
if n_nan > 0:
    nan_per_col = np.isnan(X).sum(axis=(0, 1))
    offending = {feature_cols[i]: int(c) for i, c in enumerate(nan_per_col) if c > 0}
    print('Residual NaNs by column:', offending)
assert n_nan == 0, "Unexpected NaNs survived cleaning -- investigate the columns above before saving."

print("\nFL / NF balance:")
print(meta_df['fl_nf_label'].value_counts())

print("\nFlare class breakdown:")
print(meta_df['flare_class'].value_counts())

eclipse_rate = meta_df['n_interpolated_rows'].sum() / (X.shape[0] * X.shape[1])
print(f"\nOverall eclipse-interpolated row rate: {eclipse_rate:.4%}")
print("(Should land close to the ~0.21% full-partition rate established during EDA.)")

n_xrq_degraded = int(meta_df['xrquality_degraded'].sum())
print(f"\nInstances with at least one degraded-XRQUALITY row: {n_xrq_degraded} "
      f"({n_xrq_degraded / len(meta_df):.2%}) — flag these for label-trust review "
      "at training time rather than dropping them here.")

## 10. Save the Combined Dataset to Drive

In [ ]:
tensor_path = os.path.join(output_dir, 'partition1_combined.npz')
cols_path = os.path.join(output_dir, 'partition1_feature_columns.json')
meta_path = os.path.join(output_dir, 'partition1_metadata.csv')
skipped_path = os.path.join(output_dir, 'partition1_skipped_files.csv')

np.savez_compressed(
    tensor_path,
    X=X,
    fl_nf_label=meta_df['fl_nf_label'].to_numpy(),
    flare_class=meta_df['flare_class'].to_numpy(),
    harpnum=meta_df['harpnum'].to_numpy(),
    source_file=meta_df['source_file'].to_numpy(),
    n_interpolated_rows=meta_df['n_interpolated_rows'].to_numpy(),
    xrquality_degraded=meta_df['xrquality_degraded'].to_numpy(),
)

with open(cols_path, 'w') as f:
    json.dump(feature_cols, f, indent=2)

meta_df.to_csv(meta_path, index=False)
pd.DataFrame(skipped).to_csv(skipped_path, index=False)

print("Saved:")
print(' -', tensor_path)
print(' -', cols_path)
print(' -', meta_path)
print(' -', skipped_path)

## 11. Summary & Next Steps

`partition1_combined.npz` now holds the full Partition 1 dataset as one tensor (`X`, shape `(N, 60, F)`) plus parallel side-car arrays (`fl_nf_label`, `flare_class`, `harpnum`, `source_file`, `n_interpolated_rows`, `xrquality_degraded`). Reload it with:

```python
data = np.load('partition1_combined.npz', allow_pickle=True)
X = data['X']
fl_nf_label = data['fl_nf_label']
```

**Carried-over principles to respect from here on:**

- **Never split randomly across files.** Consecutive windows from the same active region overlap ~92% of rows — split by `harpnum` (or by partition) to avoid data leakage between train/test.
- **Use TSS, HSS, or balanced accuracy**, not raw accuracy, given the class imbalance (~5% positive naturally, ~13.5% under climatology-preserving undersampling).
- Any undersampling toward the ~13.5% positive rate should happen **after** this combined dataset exists, at the train/eval split stage — not baked into this file, so the saved `.npz` always reflects the true population.
- `xrquality_degraded` and `n_interpolated_rows` are there for you to filter/weight on later — they were deliberately *not* used to drop instances here.
- Re-run this notebook (with updated `archive_name` / `archive_url`) for Partitions 2–5 once you're ready to scale beyond Partition 1, then concatenate the resulting tensors along axis 0 — keep the per-partition metadata so you can still split by partition if you want held-out partitions for testing.